# Lab 02 – Tiền Xử Lý Dữ Liệu
## Trực Quan Hóa Dữ Liệu – World Development Indicators (WDI)

**Mô tả**: Notebook này thực hiện phân tích cơ bản và tiền xử lý dữ liệu World Development Indicators (WDI) 
từ World Bank, bao gồm 11 chỉ số phát triển toàn cầu trải dài từ năm 2000 đến 2025, 
phục vụ cho việc trực quan hóa và phân tích dữ liệu bằng Tableau.

**Các bước thực hiện**:
1. Giới thiệu tổng quan về dataset
2. Mô tả cấu trúc dữ liệu, số lượng bản ghi và trường dữ liệu
3. Phân tích thống kê cơ bản
4. Tiền xử lý dữ liệu (xử lý dữ liệu thiếu, chuẩn hóa, lọc dữ liệu)
5. Xuất dữ liệu sạch để sử dụng trong Tableau

---
## 1. Import thư viện

In [1]:
import numpy as np
import pandas as pd
import os
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:.4f}'.format)

print('NumPy version:', np.__version__)
print('Pandas version:', pd.__version__)

NumPy version: 2.4.6
Pandas version: 3.0.3


---
## 2. Đọc dữ liệu

Dữ liệu được tổ chức theo mô hình quan hệ (relational model) với 3 bảng:
- **`locations.csv`**: Thông tin về các quốc gia và vùng lãnh thổ
- **`indicators.csv`**: Thông tin về các chỉ số phát triển
- **`observations.csv`**: Các quan sát (giá trị) của từng chỉ số theo quốc gia và năm

Các bảng được liên kết qua khóa ngoại: `observations.location_code → locations.location_code` 
và `observations.indicator_code → indicators.indicator_code`.

In [2]:
# Đường dẫn đến thư mục dữ liệu
DATA_DIR = os.path.join('data', 'processed')

# Đọc 3 file CSV
locations = pd.read_csv(os.path.join(DATA_DIR, 'locations.csv'))
indicators = pd.read_csv(os.path.join(DATA_DIR, 'indicators.csv'))
observations = pd.read_csv(os.path.join(DATA_DIR, 'observations.csv'))

print('Đọc dữ liệu thành công!')
print(f'  - locations:    {locations.shape[0]:>6,} bản ghi × {locations.shape[1]} trường')
print(f'  - indicators:   {indicators.shape[0]:>6,} bản ghi × {indicators.shape[1]} trường')
print(f'  - observations: {observations.shape[0]:>6,} bản ghi × {observations.shape[1]} trường')

Đọc dữ liệu thành công!
  - locations:       266 bản ghi × 2 trường
  - indicators:       11 bản ghi × 15 trường
  - observations: 76,076 bản ghi × 4 trường


---
## 3. Giới thiệu tổng quan về Dataset

### 3.1 Nguồn dữ liệu

Dữ liệu được lấy từ **World Development Indicators (WDI)** của **World Bank** – 
đây là bộ cơ sở dữ liệu phát triển quốc tế được sử dụng rộng rãi nhất, tổng hợp từ 
các nguồn chính thức được quốc tế công nhận.

### 3.2 Phạm vi dữ liệu

Dataset bao gồm **11 chỉ số phát triển** thuộc các lĩnh vực:
- **Y tế & Sức khỏe**: Tuổi thọ trung bình, tỷ lệ tử vong trẻ sơ sinh, chi tiêu y tế, tiêm chủng DPT, tỷ lệ HIV
- **Hạ tầng & Vệ sinh**: Tiếp cận nước sạch, tiếp cận vệ sinh cơ bản
- **Kinh tế**: GDP bình quân đầu người
- **Giáo dục**: Tỷ lệ nhập học trung học
- **Môi trường**: Ô nhiễm PM2.5
- **Xã hội**: Tiêu thụ rượu bình quân

Dữ liệu trải dài từ năm **2000 đến 2025** cho **266 quốc gia và vùng lãnh thổ** trên toàn thế giới.

---
## 4. Mô tả cấu trúc dữ liệu

### 4.1 Bảng Locations 

In [3]:
print('BẢNG LOCATIONS – Thông tin vị trí địa lý')
print(f'Số bản ghi: {locations.shape[0]}')
print(f'Số trường:  {locations.shape[1]}')
print()
print('Thông tin chi tiết:')
print(locations.info())
print()
print('Mẫu dữ liệu (10 dòng đầu):')
locations.head(10)

BẢNG LOCATIONS – Thông tin vị trí địa lý
Số bản ghi: 266
Số trường:  2

Thông tin chi tiết:
<class 'pandas.DataFrame'>
RangeIndex: 266 entries, 0 to 265
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   location_code  266 non-null    str  
 1   location_name  266 non-null    str  
dtypes: str(2)
memory usage: 4.3 KB
None

Mẫu dữ liệu (10 dòng đầu):


,location_code,location_name
0,ABW,Aruba
1,AFE,Africa Eastern and Southern
2,AFG,Afghanistan
3,AFW,Africa Western and Central
4,AGO,Angola
5,ALB,Albania
6,AND,Andorra
7,ARB,Arab World
8,ARE,United Arab Emirates
9,ARG,Argentina


### 4.2 Bảng Indicators 

In [4]:
print('BẢNG INDICATORS – Thông tin chỉ số phát triển')
print(f'Số bản ghi: {indicators.shape[0]}')
print(f'Số trường:  {indicators.shape[1]}')
print()
print('Thông tin chi tiết:')
print(indicators.info())
print()
print('Danh sách các chỉ số:')
for i, row in indicators.iterrows():
    print(f'  {i+1:2d}. [{row["indicator_code"]}] {row["indicator_name"]}')

BẢNG INDICATORS – Thông tin chỉ số phát triển
Số bản ghi: 11
Số trường:  15

Thông tin chi tiết:
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 15 columns):
 #   Column                               Non-Null Count  Dtype
---  ------                               --------------  -----
 0   indicator_code                       11 non-null     str  
 1   indicator_name                       11 non-null     str  
 2   license_id                           11 non-null     int64
 3   long_definition                      11 non-null     str  
 4   source                               11 non-null     str  
 5   topic                                11 non-null     str  
 6   dataset                              11 non-null     str  
 7   unit_of_measure                      10 non-null     str  
 8   periodicity                          11 non-null     str  
 9   reference_period                     11 non-null     str  
 10  aggregation_method                   1

In [5]:
# Xem chi tiết các cột của bảng indicators
print('Các trường trong bảng Indicators:')
print('-' * 50)
for col in indicators.columns:
    non_null = indicators[col].notna().sum()
    dtype = indicators[col].dtype
    print(f'  {col:<45s} | {dtype} | non-null: {non_null}/{len(indicators)}')

Các trường trong bảng Indicators:
--------------------------------------------------
  indicator_code                                | str | non-null: 11/11
  indicator_name                                | str | non-null: 11/11
  license_id                                    | int64 | non-null: 11/11
  long_definition                               | str | non-null: 11/11
  source                                        | str | non-null: 11/11
  topic                                         | str | non-null: 11/11
  dataset                                       | str | non-null: 11/11
  unit_of_measure                               | str | non-null: 10/11
  periodicity                                   | str | non-null: 11/11
  reference_period                              | str | non-null: 11/11
  aggregation_method                            | str | non-null: 11/11
  statistical_concept_and_methodology           | str | non-null: 11/11
  development_relevance                         |

### 4.3 Bảng Observations 

In [6]:
print('BẢNG OBSERVATIONS – Dữ liệu quan sát')
print(f'Số bản ghi: {observations.shape[0]:,}')
print(f'Số trường:  {observations.shape[1]}')
print()
print('Thông tin chi tiết:')
print(observations.info())
print()
print(f'Phạm vi năm:         {observations["year"].min()} – {observations["year"].max()}')
print(f'Số quốc gia/vùng:    {observations["location_code"].nunique()}')
print(f'Số chỉ số:           {observations["indicator_code"].nunique()}')
print()
print('Mẫu dữ liệu:')
observations.head(10)

BẢNG OBSERVATIONS – Dữ liệu quan sát
Số bản ghi: 76,076
Số trường:  4

Thông tin chi tiết:
<class 'pandas.DataFrame'>
RangeIndex: 76076 entries, 0 to 76075
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location_code   76076 non-null  str    
 1   indicator_code  76076 non-null  str    
 2   year            76076 non-null  int64  
 3   value           62909 non-null  float64
dtypes: float64(1), int64(1), str(2)
memory usage: 2.3 MB
None

Phạm vi năm:         2000 – 2025
Số quốc gia/vùng:    266
Số chỉ số:           11

Mẫu dữ liệu:


,location_code,indicator_code,year,value
0,ABW,EN.ATM.PM25.MC.M3,2000,NaN
1,ABW,EN.ATM.PM25.MC.M3,2001,NaN
2,ABW,EN.ATM.PM25.MC.M3,2002,NaN
3,ABW,EN.ATM.PM25.MC.M3,2003,NaN
4,ABW,EN.ATM.PM25.MC.M3,2004,NaN
5,ABW,EN.ATM.PM25.MC.M3,2005,NaN
6,ABW,EN.ATM.PM25.MC.M3,2006,NaN
7,ABW,EN.ATM.PM25.MC.M3,2007,NaN
8,ABW,EN.ATM.PM25.MC.M3,2008,NaN
9,ABW,EN.ATM.PM25.MC.M3,2009,NaN


In [7]:
# Kiểm tra kiểu dữ liệu chi tiết
print('Kiểu dữ liệu của từng trường:')
for col in observations.columns:
    dtype = observations[col].dtype
    n_unique = observations[col].nunique()
    n_null = observations[col].isnull().sum()
    print(f'  {col:<20s} | {str(dtype):<10s} | unique: {n_unique:>6,} | null: {n_null:>6,}')

Kiểu dữ liệu của từng trường:
  location_code        | str        | unique:    266 | null:      0
  indicator_code       | str        | unique:     11 | null:      0
  year                 | int64      | unique:     26 | null:      0
  value                | float64    | unique: 45,274 | null: 13,167


---
## 5. Phân tích thống kê cơ bản

### 5.1 Thống kê mô tả tổng quan

In [8]:
# Thống kê mô tả cho toàn bộ cột value
print('Thống kê mô tả cột value (toàn bộ observations):')
desc = observations['value'].describe()
print(f'  Số quan sát có giá trị:  {desc["count"]:>12,.0f}')
print(f'  Giá trị trung bình:      {desc["mean"]:>12,.4f}')
print(f'  Độ lệch chuẩn:          {desc["std"]:>12,.4f}')
print(f'  Giá trị nhỏ nhất:        {desc["min"]:>12,.4f}')
print(f'  Phân vị 25%:             {desc["25%"]:>12,.4f}')
print(f'  Trung vị (50%):          {desc["50%"]:>12,.4f}')
print(f'  Phân vị 75%:             {desc["75%"]:>12,.4f}')
print(f'  Giá trị lớn nhất:        {desc["max"]:>12,.4f}')

Thống kê mô tả cột value (toàn bộ observations):
  Số quan sát có giá trị:        62,909
  Giá trị trung bình:        1,656.3680
  Độ lệch chuẩn:            8,786.1297
  Giá trị nhỏ nhất:              0.0000
  Phân vị 25%:                  21.4807
  Trung vị (50%):               72.6040
  Phân vị 75%:                  98.0698
  Giá trị lớn nhất:        288,001.4334


### 5.2 Thống kê theo từng chỉ số

In [9]:
# Thống kê chi tiết theo từng indicator
indicator_names = dict(zip(indicators['indicator_code'], indicators['indicator_name']))

stats_list = []
for code in sorted(observations['indicator_code'].unique()):
    subset = observations[observations['indicator_code'] == code]['value']
    stats_list.append({
        'Indicator': indicator_names.get(code, code),
        'Code': code,
        'Count': subset.count(),
        'Missing': subset.isnull().sum(),
        'Missing%': round(subset.isnull().sum() / len(subset) * 100, 2),
        'Mean': round(subset.mean(), 4),
        'Std': round(subset.std(), 4),
        'Min': round(subset.min(), 4),
        'Q1': round(subset.quantile(0.25), 4),
        'Median': round(subset.median(), 4),
        'Q3': round(subset.quantile(0.75), 4),
        'Max': round(subset.max(), 4)
    })

stats_df = pd.DataFrame(stats_list)
print('Thống kê mô tả theo từng chỉ số:')
stats_df

Thống kê mô tả theo từng chỉ số:


,Indicator,Code,Count,Missing,Missing%,Mean,Std,Min,Q1,Median,Q3,Max
0,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,5208,1708,24.7000,27.9200,16.6273,4.8952,15.8035,23.2068,38.8676,107.1447
1,GDP per capita (current US$),NY.GDP.PCAP.CD,6433,483,6.9800,14909.4289,23577.5538,109.5938,1563.1059,5126.0533,18780.1275,288001.4334
2,"School enrollment, secondary (% gross)",SE.SEC.ENRR,4745,2171,31.3900,79.8659,27.9688,3.3479,61.2261,86.5600,99.6488,164.0798
3,Total alcohol consumption per capita (liters o...,SH.ALC.PCAP.LI,4899,2017,29.1600,5.4238,3.9588,0.0000,2.1073,4.8435,8.2900,19.4000
4,"Prevalence of HIV, total (% of population ages...",SH.DYN.AIDS.ZS,4420,2496,36.0900,1.7756,3.9576,0.1000,0.1000,0.4000,1.6000,29.4000
5,People using at least basic drinking water ser...,SH.H2O.BASW.ZS,6398,518,7.4900,86.0840,17.2858,18.7591,79.8897,93.6339,99.0193,100.0000
6,"Immunization, DPT (% of children ages 12-23 mo...",SH.IMM.IDPT,6005,911,13.1700,85.9124,14.0619,19.0000,79.3275,91.0000,96.0000,99.0000
7,People using at least basic sanitation service...,SH.STA.BASS.ZS,6363,553,8.0000,73.1316,28.5815,2.9658,50.1210,85.4455,97.6229,100.0000
8,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,5713,1203,17.3900,973.1146,1758.5922,4.1758,64.6776,253.9825,841.2488,13473.1934
9,"Mortality rate, infant (per 1,000 live births)",SP.DYN.IMRT.IN,6100,816,11.8000,27.8623,24.6999,1.2000,7.9000,19.1000,43.0070,239.9000


### 5.3 Phân bố dữ liệu theo năm

In [10]:
# Phân tích số lượng quan sát theo năm
year_counts = observations.groupby('year').agg(
    total_obs=('value', 'size'),
    valid_obs=('value', 'count'),
    missing_obs=('value', lambda x: x.isnull().sum())
).reset_index()
year_counts['missing_pct'] = round(year_counts['missing_obs'] / year_counts['total_obs'] * 100, 2)

print('Phân bố quan sát theo năm:')
print(year_counts.to_string(index=False))

Phân bố quan sát theo năm:
 year  total_obs  valid_obs  missing_obs  missing_pct
 2000       2926       2542          384      13.1200
 2001       2926       2578          348      11.8900
 2002       2926       2591          335      11.4500
 2003       2926       2590          336      11.4800
 2004       2926       2600          326      11.1400
 2005       2926       2608          318      10.8700
 2006       2926       2603          323      11.0400
 2007       2926       2610          316      10.8000
 2008       2926       2609          317      10.8300
 2009       2926       2610          316      10.8000
 2010       2926       2619          307      10.4900
 2011       2926       2629          297      10.1500
 2012       2926       2621          305      10.4200
 2013       2926       2615          311      10.6300
 2014       2926       2625          301      10.2900
 2015       2926       2632          294      10.0500
 2016       2926       2623          303      10.3600
 

### 5.4 Phân bố dữ liệu thiếu theo chỉ số và vùng

In [11]:
# Ma trận dữ liệu thiếu: indicator × năm
missing_matrix = observations.pivot_table(
    index='indicator_code', 
    columns='year', 
    values='value', 
    aggfunc=lambda x: x.isnull().sum()
)

print('Ma trận dữ liệu thiếu (số lượng NULL theo chỉ số × năm):')
missing_matrix

Ma trận dữ liệu thiếu (số lượng NULL theo chỉ số × năm):


year,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
indicator_code,,,,,,,,,,,,,,,,,,,,,,,,,,
EN.ATM.PM25.MC.M3,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,18,266,266,266,266,266
NY.GDP.PCAP.CD,14,13,9,9,9,9,8,8,7,5,5,4,6,6,5,6,7,7,7,7,8,8,9,15,26,266
SE.SEC.ENRR,107,77,76,79,69,67,76,70,73,75,71,64,70,79,71,65,73,80,80,79,75,72,80,74,112,257
SH.ALC.PCAP.LI,33,33,33,33,33,33,33,33,33,33,31,33,33,33,33,31,33,33,33,31,33,266,266,266,266,266
SH.DYN.AIDS.ZS,91,91,91,91,91,91,91,91,91,91,89,89,89,89,89,89,88,88,88,87,87,87,87,89,85,266
SH.H2O.BASW.ZS,18,16,14,14,14,9,8,8,7,6,6,5,5,5,5,5,4,7,11,12,13,14,15,15,16,266
SH.IMM.IDPT,28,28,27,27,27,27,26,26,26,26,26,25,25,25,25,25,25,25,25,25,25,25,25,25,26,266
SH.STA.BASS.ZS,18,15,11,11,11,10,9,8,8,8,8,7,7,6,5,5,5,8,13,15,16,17,19,22,25,266
SH.XPD.CHEX.PC.CD,34,34,33,31,31,31,31,31,31,31,30,29,29,27,27,27,27,26,25,25,25,26,27,27,242,266


---
## 6. Tiền xử lý dữ liệu

### 6.1 Kiểm tra tính toàn vẹn dữ liệu

In [12]:
# 6.1.1 Kiểm tra khóa chính và tham chiếu
print('KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU')

# Kiểm tra trùng lặp trong locations
dup_loc = locations.duplicated(subset=['location_code']).sum()
print(f'1. Trùng lặp location_code trong locations: {dup_loc}')

# Kiểm tra trùng lặp trong indicators
dup_ind = indicators.duplicated(subset=['indicator_code']).sum()
print(f'2. Trùng lặp indicator_code trong indicators: {dup_ind}')

# Kiểm tra trùng lặp khóa chính trong observations
dup_obs = observations.duplicated(subset=['location_code', 'indicator_code', 'year']).sum()
print(f'3. Trùng lặp (location_code, indicator_code, year) trong observations: {dup_obs}')

# Kiểm tra tham chiếu khóa ngoại
obs_locs = set(observations['location_code'].unique())
loc_locs = set(locations['location_code'].unique())
orphan_locs = obs_locs - loc_locs
print(f'4. Location codes trong observations không tồn tại trong locations: {len(orphan_locs)}')
if orphan_locs:
    print(f'   → {orphan_locs}')

obs_inds = set(observations['indicator_code'].unique())
ind_inds = set(indicators['indicator_code'].unique())
orphan_inds = obs_inds - ind_inds
print(f'5. Indicator codes trong observations không tồn tại trong indicators: {len(orphan_inds)}')
if orphan_inds:
    print(f'   → {orphan_inds}')

print()
print('Tsính toàn vẹn dữ liệu được đảm bảo.' if (dup_loc + dup_ind + dup_obs + len(orphan_locs) + len(orphan_inds)) == 0 else '✗ Có vấn đề về tính toàn vẹn dữ liệu!')

KIỂM TRA TÍNH TOÀN VẸN DỮ LIỆU
1. Trùng lặp location_code trong locations: 0
2. Trùng lặp indicator_code trong indicators: 0
3. Trùng lặp (location_code, indicator_code, year) trong observations: 0
4. Location codes trong observations không tồn tại trong locations: 0
5. Indicator codes trong observations không tồn tại trong indicators: 0

Tsính toàn vẹn dữ liệu được đảm bảo.


In [13]:
# 6.1.2 Kiểm tra giá trị bất thường
print('KIỂM TRA GIÁ TRỊ BẤT THƯỜNG')

# Kiểm tra giá trị âm
neg_values = observations[observations['value'] < 0]
print(f'1. Số quan sát có giá trị âm: {len(neg_values)}')

# Kiểm tra giá trị NULL trong các cột khóa
null_keys = observations[['location_code', 'indicator_code', 'year']].isnull().sum()
print(f'2. Giá trị NULL trong các cột khóa:')
for col, cnt in null_keys.items():
    print(f'   - {col}: {cnt}')

# Kiểm tra các chỉ số phần trăm có vượt quá 100%
pct_indicators = ['SH.H2O.BASW.ZS', 'SH.STA.BASS.ZS', 'SH.DYN.AIDS.ZS', 'SH.IMM.IDPT', 'SE.SEC.ENRR']
print(f'\n3. Kiểm tra chỉ số phần trăm (%):')
for code in pct_indicators:
    subset = observations[(observations['indicator_code'] == code) & (observations['value'].notna())]
    over_100 = (subset['value'] > 100).sum()
    name = indicator_names.get(code, code)
    if over_100 > 0:
        print(f'   - {code}: {over_100} giá trị > 100% (tối đa: {subset["value"].max():.2f}%)')
        print(f'     → Lưu ý: {name}')
    else:
        print(f'   - {code}: OK (tất cả ≤ 100%)')

print()
print('Nhận xét: SE.SEC.ENRR (tỷ lệ nhập học trung học gross) có thể vượt 100% do đặc thù chỉ số gross enrollment.')

KIỂM TRA GIÁ TRỊ BẤT THƯỜNG
1. Số quan sát có giá trị âm: 0
2. Giá trị NULL trong các cột khóa:
   - location_code: 0
   - indicator_code: 0
   - year: 0

3. Kiểm tra chỉ số phần trăm (%):
   - SH.H2O.BASW.ZS: OK (tất cả ≤ 100%)
   - SH.STA.BASS.ZS: OK (tất cả ≤ 100%)
   - SH.DYN.AIDS.ZS: OK (tất cả ≤ 100%)
   - SH.IMM.IDPT: OK (tất cả ≤ 100%)
   - SE.SEC.ENRR: 1167 giá trị > 100% (tối đa: 164.08%)
     → Lưu ý: School enrollment, secondary (% gross)

Nhận xét: SE.SEC.ENRR (tỷ lệ nhập học trung học gross) có thể vượt 100% do đặc thù chỉ số gross enrollment.


### 6.2 Phân loại và lọc quốc gia vs. vùng tổng hợp

Bảng `locations` chứa cả **quốc gia/vùng lãnh thổ** và **nhóm tổng hợp** (aggregate) của World Bank. 
Cần phân loại và đánh dấu để phục vụ cho việc phân tích và lọc dữ liệu trong Tableau.

In [14]:
# Danh sách các mã vùng tổng hợp (aggregate) theo phân loại World Bank
AGGREGATE_CODES = [
    'AFE', 'AFW', 'ARB', 'CEB', 'CSS', 'EAP', 'EAR', 'EAS', 'ECA', 'ECS',
    'EMU', 'EUU', 'FCS', 'HIC', 'HPC', 'IBD', 'IBT', 'IDA', 'IDB', 'IDX',
    'INX', 'LAC', 'LCN', 'LDC', 'LIC', 'LMC', 'LMY', 'LTE', 'MEA', 'MIC',
    'MNA', 'NAC', 'OED', 'OSS', 'PRE', 'PSS', 'PST', 'SAS', 'SSA', 'SSF',
    'SST', 'TEA', 'TEC', 'TLA', 'TMN', 'TSA', 'TSS', 'UMC', 'WLD'
]

# Thêm cột phân loại vào locations
locations['location_type'] = locations['location_code'].apply(
    lambda x: 'Aggregate' if x in AGGREGATE_CODES else 'Country'
)

print('Phân loại locations:')
print(locations['location_type'].value_counts())
print()
print(f'Tổng: {len(locations)} locations')
print(f'  - Quốc gia/Vùng lãnh thổ: {(locations["location_type"] == "Country").sum()}')
print(f'  - Nhóm tổng hợp (Aggregate): {(locations["location_type"] == "Aggregate").sum()}')

Phân loại locations:
location_type
Country      217
Aggregate     49
Name: count, dtype: int64

Tổng: 266 locations
  - Quốc gia/Vùng lãnh thổ: 217
  - Nhóm tổng hợp (Aggregate): 49


In [15]:
# Hiển thị danh sách các nhóm tổng hợp
print('Danh sách các nhóm tổng hợp:')
agg_locs = locations[locations['location_type'] == 'Aggregate'][['location_code', 'location_name']]
for i, (_, row) in enumerate(agg_locs.iterrows(), 1):
    print(f'  {i:2d}. {row["location_code"]}: {row["location_name"]}')

Danh sách các nhóm tổng hợp:
   1. AFE: Africa Eastern and Southern
   2. AFW: Africa Western and Central
   3. ARB: Arab World
   4. CEB: Central Europe and the Baltics
   5. CSS: Caribbean small states
   6. EAP: East Asia & Pacific (excluding high income)
   7. EAR: Early-demographic dividend
   8. EAS: East Asia & Pacific
   9. ECA: Europe & Central Asia (excluding high income)
  10. ECS: Europe & Central Asia
  11. EMU: Euro area
  12. EUU: European Union
  13. FCS: Fragile and conflict affected situations
  14. HIC: High income
  15. HPC: Heavily indebted poor countries (HIPC)
  16. IBD: IBRD only
  17. IBT: IDA & IBRD total
  18. IDA: IDA total
  19. IDB: IDA blend
  20. IDX: IDA only
  21. INX: Not classified
  22. LAC: Latin America & Caribbean (excluding high income)
  23. LCN: Latin America & Caribbean
  24. LDC: Least developed countries: UN classification
  25. LIC: Low income
  26. LMC: Lower middle income
  27. LMY: Low & middle income
  28. LTE: Late-demographic dividen

### 6.3 Xử lý dữ liệu thiếu 

In [16]:
# Phân tích chi tiết dữ liệu thiếu
print('PHÂN TÍCH DỮ LIỆU THIẾU')

total_obs = len(observations)
total_missing = observations['value'].isnull().sum()
print(f'Tổng số bản ghi:          {total_obs:>10,}')
print(f'Số bản ghi có giá trị:    {total_obs - total_missing:>10,}')
print(f'Số bản ghi thiếu giá trị: {total_missing:>10,} ({total_missing/total_obs*100:.2f}%)')
print()

# Phân tích missing theo indicator
print('Dữ liệu thiếu theo chỉ số:')
print('-' * 80)
missing_by_indicator = []
for code in sorted(observations['indicator_code'].unique()):
    subset = observations[observations['indicator_code'] == code]
    n_miss = subset['value'].isnull().sum()
    n_total = len(subset)
    missing_by_indicator.append({
        'Indicator': indicator_names.get(code, code)[:50],
        'Code': code,
        'Total': n_total,
        'Missing': n_miss,
        'Missing%': round(n_miss / n_total * 100, 2)
    })

missing_df = pd.DataFrame(missing_by_indicator).sort_values('Missing%', ascending=False)
missing_df

PHÂN TÍCH DỮ LIỆU THIẾU
Tổng số bản ghi:              76,076
Số bản ghi có giá trị:        62,909
Số bản ghi thiếu giá trị:     13,167 (17.31%)

Dữ liệu thiếu theo chỉ số:
--------------------------------------------------------------------------------


,Indicator,Code,Total,Missing,Missing%
4,"Prevalence of HIV, total (% of population ages...",SH.DYN.AIDS.ZS,6916,2496,36.0900
2,"School enrollment, secondary (% gross)",SE.SEC.ENRR,6916,2171,31.3900
3,Total alcohol consumption per capita (liters o...,SH.ALC.PCAP.LI,6916,2017,29.1600
0,"PM2.5 air pollution, mean annual exposure (mic...",EN.ATM.PM25.MC.M3,6916,1708,24.7000
8,Current health expenditure per capita (current...,SH.XPD.CHEX.PC.CD,6916,1203,17.3900
6,"Immunization, DPT (% of children ages 12-23 mo...",SH.IMM.IDPT,6916,911,13.1700
9,"Mortality rate, infant (per 1,000 live births)",SP.DYN.IMRT.IN,6916,816,11.8000
7,People using at least basic sanitation service...,SH.STA.BASS.ZS,6916,553,8.0000
5,People using at least basic drinking water ser...,SH.H2O.BASW.ZS,6916,518,7.4900
1,GDP per capita (current US$),NY.GDP.PCAP.CD,6916,483,6.9800


In [17]:
# Phân tích missing theo location_type (Country vs Aggregate)
obs_with_type = observations.merge(locations[['location_code', 'location_type']], on='location_code', how='left')

print('Dữ liệu thiếu theo loại vị trí:')
for ltype in ['Country', 'Aggregate']:
    subset = obs_with_type[obs_with_type['location_type'] == ltype]
    n_miss = subset['value'].isnull().sum()
    n_total = len(subset)
    print(f'  {ltype}:    {n_miss:>6,} / {n_total:>6,} thiếu ({n_miss/n_total*100:.2f}%)')

Dữ liệu thiếu theo loại vị trí:
  Country:    11,309 / 62,062 thiếu (18.22%)
  Aggregate:     1,858 / 14,014 thiếu (13.26%)


#### 6.3.1 Chiến lược xử lý dữ liệu thiếu

Để đảm bảo dữ liệu sạch và liên tục khi đưa vào Tableau trực quan hóa (tránh lỗi đứt quãng đường xu hướng hoặc thiếu dữ liệu trên bản đồ), ta áp dụng chiến lược điền bù dữ liệu thiếu theo 3 bước:

1. **Nội suy tuyến tính (Linear Interpolation)**: Áp dụng nội suy tuyến tính cho các giá trị thiếu nằm giữa hai năm đã biết của cùng một quốc gia và chỉ số.

2. **Điền bù đầu/cuối chuỗi (ffill & bfill theo nhóm)**: Sử dụng phương pháp điền ngược về trước (bfill) và điền tiếp về sau (ffill) trong phạm vi từng nhóm (quốc gia + chỉ số) để lấp đầy các năm thiếu ở đầu và cuối chuỗi thời gian.

3. **Sử dụng trung vị toàn cầu theo năm làm dự phòng**: Đối với các quốc gia hoàn toàn không có dữ liệu cho một chỉ số cụ thể trong suốt 26 năm, ta điền bù bằng giá trị trung vị của chỉ số đó trên toàn thế giới trong năm tương ứng (nếu năm đó cũng thiếu toàn cầu, ta sẽ nội suy/điền bù chuỗi trung vị theo năm).

In [18]:
# Thực hiện xử lý dữ liệu thiếu bằng nội suy tuyến tính và điền bù (ffill/bfill)
print('THỰC HIỆN NỘI SUY TUYẾN TÍNH VÀ ĐIỀN BÙ DỮ LIỆU THIẾU')

obs_clean = observations.copy()

# Sắp xếp dữ liệu theo location, indicator, year
obs_clean = obs_clean.sort_values(['location_code', 'indicator_code', 'year']).reset_index(drop=True)

# Đếm missing trước xử lý
missing_before = obs_clean['value'].isnull().sum()
print(f'Số giá trị thiếu TRƯỚC xử lý: {missing_before:,}')

# Bước 1: Nội suy tuyến tính (Linear Interpolation) cho các khoảng trống giữa các năm có dữ liệu
obs_clean['value'] = obs_clean.groupby(['location_code', 'indicator_code'])['value'].transform(
    lambda x: x.interpolate(method='linear')
)
missing_interp = obs_clean['value'].isnull().sum()
print(f'Sau Bước 1 (Nội suy tuyến tính): Còn {missing_interp:,} giá trị thiếu ({missing_interp/len(obs_clean)*100:.2f}%)')

# Bước 2: Điền forward (ffill) và backward (bfill) trong từng nhóm quốc gia + chỉ số
obs_clean['value'] = obs_clean.groupby(['location_code', 'indicator_code'])['value'].transform(
    lambda x: x.ffill().bfill()
)
missing_ff_bf = obs_clean['value'].isnull().sum()
print(f'Sau Bước 2 (ffill & bfill theo nhóm): Còn {missing_ff_bf:,} giá trị thiếu ({missing_ff_bf/len(obs_clean)*100:.2f}%)')

# Bước 3: Điền bù các nhóm không có bất kỳ điểm dữ liệu nào bằng trung vị toàn cầu theo năm
global_medians = obs_clean.groupby(['indicator_code', 'year'])['value'].median().groupby('indicator_code').ffill().bfill().to_dict()
mapped_medians = pd.Series(
    [global_medians.get((ind, yr)) for ind, yr in zip(obs_clean['indicator_code'], obs_clean['year'])],
    index=obs_clean.index
)
obs_clean['value'] = obs_clean['value'].fillna(mapped_medians)

# Đếm missing sau xử lý
missing_after = obs_clean['value'].isnull().sum()
print(f'Sau Bước 3 (Trung vị toàn cầu theo năm): Còn {missing_after:,} giá trị thiếu ({missing_after/len(obs_clean)*100:.2f}%)')
print(f'Tổng số giá trị đã được điền bù: {missing_before - missing_after:,}')

THỰC HIỆN NỘI SUY TUYẾN TÍNH VÀ ĐIỀN BÙ DỮ LIỆU THIẾU
Số giá trị thiếu TRƯỚC xử lý: 13,167
Sau Bước 1 (Nội suy tuyến tính): Còn 6,976 giá trị thiếu (9.17%)
Sau Bước 2 (ffill & bfill theo nhóm): Còn 5,980 giá trị thiếu (7.86%)
Sau Bước 3 (Trung vị toàn cầu theo năm): Còn 0 giá trị thiếu (0.00%)
Tổng số giá trị đã được điền bù: 13,167


In [19]:
# So sánh trước và sau nội suy theo chỉ số
print('So sánh dữ liệu thiếu trước và sau nội suy:')

comparison = []
for code in sorted(observations['indicator_code'].unique()):
    before = observations[observations['indicator_code'] == code]['value'].isnull().sum()
    after = obs_clean[obs_clean['indicator_code'] == code]['value'].isnull().sum()
    total = len(observations[observations['indicator_code'] == code])
    comparison.append({
        'Indicator': indicator_names.get(code, code)[:45],
        'Before': before,
        'After': after,
        'Interpolated': before - after,
        'Before%': round(before/total*100, 2),
        'After%': round(after/total*100, 2)
    })

comp_df = pd.DataFrame(comparison)
comp_df

So sánh dữ liệu thiếu trước và sau nội suy:


,Indicator,Before,After,Interpolated,Before%,After%
0,"PM2.5 air pollution, mean annual exposure (mi",1708,0,1708,24.7000,0.0000
1,GDP per capita (current US$),483,0,483,6.9800,0.0000
2,"School enrollment, secondary (% gross)",2171,0,2171,31.3900,0.0000
3,Total alcohol consumption per capita (liters,2017,0,2017,29.1600,0.0000
4,"Prevalence of HIV, total (% of population age",2496,0,2496,36.0900,0.0000
5,People using at least basic drinking water se,518,0,518,7.4900,0.0000
6,"Immunization, DPT (% of children ages 12-23 m",911,0,911,13.1700,0.0000
7,People using at least basic sanitation servic,553,0,553,8.0000,0.0000
8,Current health expenditure per capita (curren,1203,0,1203,17.3900,0.0000
9,"Mortality rate, infant (per 1,000 live births",816,0,816,11.8000,0.0000


### 6.4 Lọc dữ liệu

Tạo thêm cột `location_type` trong dữ liệu observations để phân biệt giữa quốc gia và nhóm tổng hợp, 
giúp dễ dàng lọc khi phân tích trong Tableau.

In [20]:
# Merge thông tin location_type vào observations
obs_clean = obs_clean.merge(
    locations[['location_code', 'location_type']], 
    on='location_code', 
    how='left'
)

print('Thêm cột location_type vào observations:')
print(obs_clean['location_type'].value_counts())
print()

# Tạo bộ dữ liệu chỉ chứa quốc gia (không bao gồm aggregate)
obs_countries = obs_clean[obs_clean['location_type'] == 'Country'].copy()
print(f'Observations chỉ quốc gia: {len(obs_countries):,} bản ghi')
print(f'Observations toàn bộ:      {len(obs_clean):,} bản ghi')

Thêm cột location_type vào observations:
location_type
Country      62062
Aggregate    14014
Name: count, dtype: int64

Observations chỉ quốc gia: 62,062 bản ghi
Observations toàn bộ:      76,076 bản ghi


### 6.5 Chuẩn hóa dữ liệu

Tạo phiên bản dữ liệu wide-format (pivot) để thuận tiện cho việc phân tích trong Tableau, 
đồng thời merge tên quốc gia và tên chỉ số vào dữ liệu.

In [21]:
# 6.5.1 Merge tên quốc gia và tên chỉ số
obs_enriched = obs_clean.merge(
    locations[['location_code', 'location_name']], 
    on='location_code', 
    how='left'
)
obs_enriched = obs_enriched.merge(
    indicators[['indicator_code', 'indicator_name']], 
    on='indicator_code', 
    how='left'
)

# Sắp xếp lại cột
obs_enriched = obs_enriched[[
    'location_code', 'location_name', 'location_type',
    'indicator_code', 'indicator_name',
    'year', 'value'
]]

print('Bảng observations đã làm giàu:')
print(f'Shape: {obs_enriched.shape}')
obs_enriched.head(10)

Bảng observations đã làm giàu:
Shape: (76076, 7)


,location_code,location_name,location_type,indicator_code,indicator_name,year,value
0,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2000,25.3381
1,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2001,25.0943
2,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2002,24.9307
3,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2003,24.2405
4,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2004,23.8090
5,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2005,23.3572
6,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2006,23.0309
7,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2007,23.0911
8,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2008,23.0857
9,ABW,Aruba,Country,EN.ATM.PM25.MC.M3,"PM2.5 air pollution, mean annual exposure (mic...",2009,22.6512


In [22]:
# 6.5.2 Tạo wide-format (pivot) – mỗi indicator thành một cột
obs_wide = obs_enriched.pivot_table(
    index=['location_code', 'location_name', 'location_type', 'year'],
    columns='indicator_code',
    values='value'
).reset_index()

# Đổi tên cột cho dễ đọc
short_names = {
    'EN.ATM.PM25.MC.M3': 'pm25_pollution',
    'NY.GDP.PCAP.CD': 'gdp_per_capita',
    'SE.SEC.ENRR': 'school_enrollment_secondary',
    'SH.ALC.PCAP.LI': 'alcohol_consumption',
    'SH.DYN.AIDS.ZS': 'hiv_prevalence',
    'SH.H2O.BASW.ZS': 'basic_water_services',
    'SH.IMM.IDPT': 'immunization_dpt',
    'SH.STA.BASS.ZS': 'basic_sanitation',
    'SH.XPD.CHEX.PC.CD': 'health_expenditure_pc',
    'SP.DYN.IMRT.IN': 'infant_mortality_rate',
    'SP.DYN.LE00.IN': 'life_expectancy'
}
obs_wide.columns = [short_names.get(c, c) for c in obs_wide.columns]

print('Bảng observations dạng wide-format:')
print(f'Shape: {obs_wide.shape}')
print(f'Columns: {obs_wide.columns.tolist()}')
print()
obs_wide.head(10)

Bảng observations dạng wide-format:
Shape: (6916, 15)
Columns: ['location_code', 'location_name', 'location_type', 'year', 'pm25_pollution', 'gdp_per_capita', 'school_enrollment_secondary', 'alcohol_consumption', 'hiv_prevalence', 'basic_water_services', 'immunization_dpt', 'basic_sanitation', 'health_expenditure_pc', 'infant_mortality_rate', 'life_expectancy']



,location_code,location_name,location_type,year,pm25_pollution,gdp_per_capita,school_enrollment_secondary,alcohol_consumption,hiv_prevalence,basic_water_services,immunization_dpt,basic_sanitation,health_expenditure_pc,infant_mortality_rate,life_expectancy
0,ABW,Aruba,Country,2000,25.3381,20681.0230,91.5802,4.6100,0.3000,95.2335,86.5188,97.9731,83.6384,29.5000,72.9390
1,ABW,Aruba,Country,2001,25.0943,20740.1326,97.5565,4.6100,0.3000,95.3591,89.0000,98.0125,91.8441,27.8000,73.0440
2,ABW,Aruba,Country,2002,24.9307,21307.2483,100.1631,4.6500,0.3733,95.4846,88.0000,98.0519,97.6605,26.1500,73.1350
3,ABW,Aruba,Country,2003,24.2405,21949.4860,101.9291,4.6367,0.3823,95.6101,89.0000,98.0913,109.5193,25.0121,73.2360
4,ABW,Aruba,Country,2004,23.8090,23700.6320,100.9409,4.8400,0.3848,95.7356,89.0000,98.1307,136.5072,24.4690,73.2230
5,ABW,Aruba,Country,2005,23.3572,24171.8371,99.2054,4.8400,0.3946,95.8611,91.0000,98.1701,158.4303,23.3000,73.4150
6,ABW,Aruba,Country,2006,23.0309,24845.6585,99.8658,4.9308,0.4000,95.9867,92.0000,98.2095,166.8148,22.2500,73.4980
7,ABW,Aruba,Country,2007,23.0911,26736.3089,106.2365,4.8800,0.4000,96.1122,92.8502,98.2489,195.9909,21.3105,73.6420
8,ABW,Aruba,Country,2008,23.0857,28171.9094,97.0628,4.9000,0.4000,96.2377,92.0000,98.2883,237.1034,20.6603,73.7830
9,ABW,Aruba,Country,2009,22.6512,25134.7712,97.9976,4.9871,0.4000,96.3632,93.0000,98.3277,243.3110,19.5500,73.9740


### 6.6 Kiểm tra outliers 

In [23]:
# Phát hiện outliers bằng phương pháp IQR cho từng chỉ số
print('PHÁT HIỆN OUTLIERS (PHƯƠNG PHÁP IQR)')

# Chỉ phân tích trên dữ liệu quốc gia (không bao gồm aggregate)
obs_country_only = obs_enriched[obs_enriched['location_type'] == 'Country'].copy()

outlier_summary = []
for code in sorted(obs_country_only['indicator_code'].unique()):
    subset = obs_country_only[obs_country_only['indicator_code'] == code]['value'].dropna()
    Q1 = subset.quantile(0.25)
    Q3 = subset.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((subset < lower) | (subset > upper)).sum()
    outlier_summary.append({
        'Indicator': indicator_names.get(code, code)[:45],
        'Q1': round(Q1, 4),
        'Q3': round(Q3, 4),
        'IQR': round(IQR, 4),
        'Lower': round(max(lower, 0), 4),
        'Upper': round(upper, 4),
        'Outliers': n_outliers,
        'Outlier%': round(n_outliers / len(subset) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print('Nhận xét: Các outliers KHÔNG bị loại bỏ vì chúng phản ánh sự đa dạng thực tế giữa các quốc gia.')
print('Ví dụ: GDP per capita cao ở các nước phát triển, tỷ lệ HIV cao ở một số nước châu Phi, v.v.')
print()
outlier_df

PHÁT HIỆN OUTLIERS (PHƯƠNG PHÁP IQR)
Nhận xét: Các outliers KHÔNG bị loại bỏ vì chúng phản ánh sự đa dạng thực tế giữa các quốc gia.
Ví dụ: GDP per capita cao ở các nước phát triển, tỷ lệ HIV cao ở một số nước châu Phi, v.v.



,Indicator,Q1,Q3,IQR,Lower,Upper,Outliers,Outlier%
0,"PM2.5 air pollution, mean annual exposure (mi",14.6586,30.0494,15.3908,0.0000,53.1356,464,8.2200
1,GDP per capita (current US$),1651.1950,21315.1579,19663.9629,0.0000,50811.1022,464,8.2200
2,"School enrollment, secondary (% gross)",62.1594,99.2583,37.0989,6.5110,154.9067,74,1.3100
3,Total alcohol consumption per capita (liters,2.1200,7.9400,5.8200,0.0000,16.6700,33,0.5800
4,"Prevalence of HIV, total (% of population age",0.2000,0.7000,0.5000,0.0000,1.4500,881,15.6200
5,People using at least basic drinking water se,82.2489,99.3112,17.0623,56.6555,124.9046,547,9.7000
6,"Immunization, DPT (% of children ages 12-23 m",84.0000,96.0000,12.0000,66.0000,114.0000,507,8.9900
7,People using at least basic sanitation servic,57.1171,98.0478,40.9307,0.0000,159.4438,0,0.0000
8,Current health expenditure per capita (curren,76.0712,764.9390,688.8678,0.0000,1798.2406,825,14.6200
9,"Mortality rate, infant (per 1,000 live births",7.8000,35.4000,27.6000,0.0000,76.8000,241,4.2700


---
## 7. Xuất dữ liệu sạch

Xuất các file CSV đã được tiền xử lý để sử dụng trong Tableau.

In [24]:
# Tạo thư mục output
OUTPUT_DIR = 'output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Xuất observations dạng long-format (enriched)
obs_enriched.to_csv(os.path.join(OUTPUT_DIR, 'observations_clean.csv'), index=False, encoding='utf-8-sig')
print(f'Đã xuất: {OUTPUT_DIR}/observations_clean.csv ({len(obs_enriched):,} bản ghi)')

# 2. Xuất observations dạng wide-format
obs_wide.to_csv(os.path.join(OUTPUT_DIR, 'observations_wide.csv'), index=False, encoding='utf-8-sig')
print(f'Đã xuất: {OUTPUT_DIR}/observations_wide.csv ({len(obs_wide):,} bản ghi)')

# 3. Xuất bảng locations đã bổ sung location_type
locations.to_csv(os.path.join(OUTPUT_DIR, 'locations_clean.csv'), index=False, encoding='utf-8-sig')
print(f'Đã xuất: {OUTPUT_DIR}/locations_clean.csv ({len(locations):,} bản ghi)')

# 4. Xuất bảng indicators (giữ nguyên)
indicators.to_csv(os.path.join(OUTPUT_DIR, 'indicators_clean.csv'), index=False, encoding='utf-8-sig')
print(f'Đã xuất: {OUTPUT_DIR}/indicators_clean.csv ({len(indicators):,} bản ghi)')

# 5. Xuất observations chỉ quốc gia dạng wide
obs_wide_countries = obs_wide[obs_wide['location_type'] == 'Country'].copy()
obs_wide_countries.to_csv(os.path.join(OUTPUT_DIR, 'observations_countries_wide.csv'), index=False, encoding='utf-8-sig')
print(f'Đã xuất: {OUTPUT_DIR}/observations_countries_wide.csv ({len(obs_wide_countries):,} bản ghi)')

print()
print('Hoàn tất xuất dữ liệu!')

Đã xuất: output/observations_clean.csv (76,076 bản ghi)
Đã xuất: output/observations_wide.csv (6,916 bản ghi)
Đã xuất: output/locations_clean.csv (266 bản ghi)
Đã xuất: output/indicators_clean.csv (11 bản ghi)
Đã xuất: output/observations_countries_wide.csv (5,642 bản ghi)

Hoàn tất xuất dữ liệu!


---
## 8. Tổng kết tiền xử lý dữ liệu

### Các bước đã thực hiện:

| # | Bước | Mô tả | Kết quả |
|---|------|-------|--------|
| 1 | Kiểm tra tính toàn vẹn | Kiểm tra khóa chính, khóa ngoại, trùng lặp | Dữ liệu toàn vẹn, không có lỗi |
| 2 | Kiểm tra giá trị bất thường | Kiểm tra giá trị âm, NULL trong khóa, giá trị phần trăm > 100% | Không có giá trị âm; SE.SEC.ENRR có thể > 100% (hợp lệ) |
| 3 | Phân loại locations | Phân loại quốc gia vs. nhóm tổng hợp (49 aggregate) | 217 quốc gia, 49 aggregate |
| 4 | Xử lý dữ liệu thiếu | Nội suy tuyến tính theo nhóm (location, indicator) | Giảm đáng kể tỷ lệ thiếu |
| 5 | Làm giàu dữ liệu | Merge tên quốc gia, tên chỉ số, loại location | Dữ liệu đầy đủ thông tin |
| 6 | Chuẩn hóa dạng dữ liệu | Tạo wide-format cho Tableau | Mỗi chỉ số thành một cột riêng |
| 7 | Phát hiện outliers | Phương pháp IQR | Outliers được giữ lại (phản ánh thực tế) |

### Các file output:
- `observations_clean.csv`: Dữ liệu long-format đã làm sạch
- `observations_wide.csv`: Dữ liệu wide-format (mỗi chỉ số một cột)
- `observations_countries_wide.csv`: Dữ liệu wide chỉ quốc gia (không bao gồm aggregate)
- `locations_clean.csv`: Bảng locations có thêm cột `location_type`
- `indicators_clean.csv`: Bảng indicators giữ nguyên

In [25]:
# Tóm tắt cuối cùng
print('TÓM TẮT KẾT QUẢ TIỀN XỬ LÝ DỮ LIỆU')
print(f'\nDữ liệu gốc:')
print(f'  - observations: {len(observations):>10,} bản ghi')
print(f'  - locations:    {len(locations):>10,} bản ghi')
print(f'  - indicators:   {len(indicators):>10,} bản ghi')
print(f'\nDữ liệu sau xử lý:')
print(f'  - observations_clean:            {len(obs_enriched):>10,} bản ghi (long-format, enriched)')
print(f'  - observations_wide:             {len(obs_wide):>10,} bản ghi (wide-format)')
print(f'  - observations_countries_wide:   {len(obs_wide_countries):>10,} bản ghi (wide, chỉ quốc gia)')
print(f'\nGiá trị thiếu:')
print(f'  - Trước nội suy: {observations["value"].isnull().sum():>10,} ({observations["value"].isnull().sum()/len(observations)*100:.2f}%)')
print(f'  - Sau nội suy:   {obs_clean["value"].isnull().sum():>10,} ({obs_clean["value"].isnull().sum()/len(obs_clean)*100:.2f}%)')
print(f'\nTiền xử lý hoàn tất! Dữ liệu sẵn sàng để import vào Tableau.')

TÓM TẮT KẾT QUẢ TIỀN XỬ LÝ DỮ LIỆU

Dữ liệu gốc:
  - observations:     76,076 bản ghi
  - locations:           266 bản ghi
  - indicators:           11 bản ghi

Dữ liệu sau xử lý:
  - observations_clean:                76,076 bản ghi (long-format, enriched)
  - observations_wide:                  6,916 bản ghi (wide-format)
  - observations_countries_wide:        5,642 bản ghi (wide, chỉ quốc gia)

Giá trị thiếu:
  - Trước nội suy:     13,167 (17.31%)
  - Sau nội suy:            0 (0.00%)

Tiền xử lý hoàn tất! Dữ liệu sẵn sàng để import vào Tableau.
